# Clean Clinical Data

Clean the clinical data spreadsheet by:
- Standardizing ontologies
- Fix spelling, categories, etc...
- Adjust Features and Labels
- Add it to `LaminDB`

## Notebook Preferences

In [1]:
%config InlineBackend.figure_format="retina"

## Importing Libraries


In [1]:
from upath import UPath
import pandas as pd
import natsort as ns
import bionty as bt
import lamindb as ln
from lnschema_core.models import Registry

Buckaroo has been enabled as the default DataFrame viewer.  To return to default dataframe visualization use `from buckaroo import disable; disable()`
💡 connected lamindb: srivarra/nbl_assets


In [2]:
pd.set_option("future.no_silent_downcasting", True)
pd.set_option("future.infer_string", False)
pd.set_option("mode.copy_on_write", True)

In [4]:
ln.settings.transform.stem_uid = "4DLIySb5QY32"
ln.settings.transform.version = "1"
ln.settings.sync_git_repo = "https://github.com/karadavis-lab/nbl.git"
run = ln.track()
run.transform

💡 notebook imports: bionty==0.48.0 buckaroo==0.6.12 lamindb==0.75.1 lnschema_core==0.72.2 natsort==8.4.0 pandas==2.2.2 universal_pathlib==0.2.2
💡 saved: Transform(uid='4DLIySb5QY325zKv', version='1', name='Clean Clinical Data', key='00 - Clean Clinical Data', type='notebook', created_by_id=1, updated_at='2024-08-12 17:01:42 UTC')
💡 saved: Run(uid='3Lc5DOTT7khxWOQ6keYQ', transform_id=1, created_by_id=1)


Transform(uid='4DLIySb5QY325zKv', version='1', name='Clean Clinical Data', key='00 - Clean Clinical Data', type='notebook', created_by_id=1, updated_at='2024-08-12 17:01:42 UTC')

In [5]:
ln.setup.settings.instance

Current instance: srivarra/nbl_assets
- owner: srivarra
- name: nbl_assets
- storage root: /Users/srivarra/Davis Lab/neuroblastoma/nbl/data/db
- storage region: None
- db: postgresql://srivarra:***@***:5555/nbl_db
- schema: {'bionty'}
- git_repo: None

## Validate Clinical Data

### Set up Paths

In [6]:
original_data_path = UPath("../../../data/raw/original_data")
clinical_data_path: UPath = original_data_path / "Clinical Data" / "FOVs_UIDv2.xlsx"
fov_dir: UPath = original_data_path / "nbl_cohort" / "images"
label_dir: UPath = original_data_path / "nbl_cohort" / "segmentation" / "labels"

In [7]:
clinical_data = pd.read_excel(clinical_data_path)

### Standardize FOV Names

In [8]:
fovs = ns.natsorted(fov_dir.glob("*/"))

In [9]:
def convert_fov(row, fovs):
    """Adjusts the name of the FOV.

    Parameters
    ----------
    row : pd.DataFrame
        The row of the clinical data.

    Returns
    -------
    str
        The full name of the FOV.
    """
    for fov in fovs:
        if row["fov"] == fov.stem.split("-")[2]:
            return fov.stem
    return None


clinical_data["fov"] = clinical_data.apply(convert_fov, fovs=fovs, axis=1)

In [10]:
clinical_data

BuckarooWidget(buckaroo_options={'sampled': ['random'], 'auto_clean': ['aggressive', 'conservative'], 'post_pr…

### Removing Misc Column Name / Values Whitespaces


In [11]:
# strip whitespace from column names
clinical_data.columns = clinical_data.columns.str.strip()

In [12]:
for c in ["Paired sequence", "Classification of specimen", "Clinical presentation", "Risk"]:
    clinical_data[c] = clinical_data[c].str.strip()

In [13]:
bt.settings.organism = "human"

In [14]:
tissues = bt.Tissue.public()
ethnicitys = bt.Ethnicity.public()
tissues_lookup = tissues.lookup()
ethnicity_lookup = ethnicitys.lookup()

In [15]:
clinical_data["Paired sequence"] = clinical_data["Paired sequence"].map(lambda x: False if x == "No" else True)

In [16]:
clinical_data = clinical_data.replace(
    to_replace={
        "Classification of specimen": {
            "Diagnosis": "Diagnosis",
            "post-chemotherpy, local control surgery (mild paraspinal disease progression requiring laminectomy)": "Post-Chemotherapy",
            "post-chemotherapy (local control surgery, 4 cycles of ANBL0531)": "Post-Chemotherapy",
            "relapse (after 2 cycles of topo/cyclo)": "Relapse",
            "Progressive disease (re-resection, s/p chemotherapy)": "Disease Progression",
            "post-chemotherapy, local control surgery (s/p 4 cycles of induction chemo per ANBL0531)": "Post-Chemotherapy",
            "post-chemotherapy (5 cycles ANBL0532)": "Post-Chemotherapy",
            "Relapsed": "Relapse",
            "CCHS, post-chemo therapy, local control surgery (7 cycles of ANBL0531, stable disease after 6 cycles and then 1 cycle of topo/cyclo)": "Post-Chemotherapy",
            "relapse, brain metastases": "Relapse",
            "post-chemotherapy, local control surgery (2nd)": "Post-Chemotherapy",
            "post-chemotherapy, local control surgery (s/p 4 cycles of induction per ANBL0531)": "Post-Chemotherapy",
            "post-chemotherapy, local control surgery (8 cyles of ANBL0531 therapy with minimal response)": "Post-Chemotherapy",
            "Diagnosis (after a period of observation)": "Diagnosis",
            "disease progression after upfront surgery (posterior mediastinum)": "Disease Progression",
            "post-chemotherapy, local control surgery": "Post-Chemotherapy",
        },
        "Sex": {s: s.strip().lower().capitalize() for s in clinical_data["Sex"].unique()},
        "Race": {
            "Black": ethnicity_lookup.african.name,
            "White": ethnicity_lookup.european.name,
            "white": ethnicity_lookup.european.name,
            "Other": ethnicity_lookup.undefined_ancestry_population.name,
            "Arabic ": ethnicity_lookup.arab.name,
            "Asian ": ethnicity_lookup.asian.name,
            "other (egyptian)": ethnicity_lookup.egyptian.name,
            "?black ": ethnicity_lookup.african.name,
            "white ": ethnicity_lookup.european.name,
        },
        "Biopsy/surgery location": {
            "abdominal mass": tissues_lookup.abdominal_segment_element.name,
            "letfy adrenal mass": tissues_lookup.left_adrenal_gland.name,
            "Right adrenal ": tissues_lookup.right_adrenal_gland.name,
            "Abdominal mass": tissues_lookup.abdominal_segment_element.name,
            "Spinal/paraspinal ": tissues_lookup.paraspinal_region.name,
            "RP mass ": tissues_lookup.retroperitoneal_space.name,
            "abdominal mass/thoracic region mass excision": tissues_lookup.thoracic_cavity_element.name,
            "abdominal mass/diagphramtic mass": tissues_lookup.diaphragm.name,
            "left adrenal tumor": tissues_lookup.left_adrenal_gland.name,
            "pelvic mass": tissues_lookup.pelvic_region_element.name,
            "abdominal tumor resection ": tissues_lookup.abdominal_segment_element.name,
            "Retroperitoneal": tissues_lookup.retroperitoneal_space.name,
            "Abdominal/Retroperitoneal": tissues_lookup.retroperitoneal_space.name,
            "Pelvic mass, s/p 2 cycles of ANBL0531, limited response to chemo with tumor growth": tissues_lookup.pelvic_region_element.name,
            "Paraspinal ": tissues_lookup.paraspinal_region.name,
            "paraspinal ": tissues_lookup.paraspinal_region.name,
            "Right Adrenal": tissues_lookup.right_adrenal_gland.name,
            "Liver": tissues_lookup.liver.name,
            "abdominal tumor": tissues_lookup.abdominal_segment_element.name,
            "Abdominal tumor, lymph nodes": tissues_lookup.abdominal_lymph_node.name,
            "Brain mets, relapse during maintenance GD2 antibody": tissues_lookup.brain.name,
            "paraspinal mass": tissues_lookup.paraspinal_region.name,
            "abdominal tumor resection": tissues_lookup.abdominal_segment_element.name,
            "right adrenal mass": tissues_lookup.right_adrenal_gland.name,
            "right adrenal gland resection ": tissues_lookup.right_adrenal_gland.name,
            "retroperitoneal mass": tissues_lookup.retroperitoneal_space.name,
            "neck mass": tissues_lookup.neck.name,
            "abdominal/paraspinal mass resection": tissues_lookup.paraspinal_region.name,
            "right chect, posterior mediastinal ": tissues_lookup.posterior_mediastinum.name,
            "retroperitoneal": tissues_lookup.retroperitoneal_space.name,
            "abdominal tumor resection after 4 cycles of ANBL0531 ": tissues_lookup.abdominal_segment_element.name,
            "right adrenal gland": tissues_lookup.right_adrenal_gland.name,
            "right apical chest mass resection": tissues_lookup.chest.name,
            "abdominal mass/liver nodule": tissues_lookup.liver.name,
            "pelvic tumor": tissues_lookup.pelvic_region_element.name,
            "right neck mass": tissues_lookup.neck.name,
            "Abd mass": tissues_lookup.abdominal_segment_element.name,
            "abdominal mass biopsy": tissues_lookup.abdominal_segment_element.name,
            "right axilla": tissues_lookup.axilla.name,
            "thoracic tumor": tissues_lookup.thoracic_cavity_element.name,
            "left chest mass": tissues_lookup.chest.name,
            "b/l adrenal masses": tissues_lookup.adrenal_tissue.name,
            "adrenalectomy": tissues_lookup.adrenal_tissue.name,
            "lefty adrenal mass": tissues_lookup.left_adrenal_gland.name,
        },
        "Risk": {
            "Intermediate": "Intermediate",
            "High": "High",
            "Inrtermediate, mild disease progression": "Intermediate",
            "intermediate": "Intermediate",
            "High (relapsed)": "High",
            "Low": "Low",
            "Low (would be IR now?)": "Low",
            "High (due to nodular ganglioneuroblastoma)": "High",
        },
    },
)

In [17]:
u_vma_hva = "Urine VMA/HVA (g/g Cr)"

In [18]:
clinical_data = clinical_data.replace(to_replace={u_vma_hva: {"n/a ": pd.NA, ">227/>227": "227/227"}})
clinical_data = clinical_data.fillna({u_vma_hva: pd.NA})

In [19]:
vma_hva_df = (
    clinical_data[u_vma_hva]
    .str.split("/", expand=True)
    .rename(columns={0: "VMA (g Cr)", 1: "HVA (g Cr)"})
    .apply(pd.to_numeric, errors="coerce")
)

In [20]:
clinical_data = clinical_data.drop(columns=[u_vma_hva])

In [21]:
clinical_data = pd.concat([clinical_data, vma_hva_df], axis=1)

In [22]:
hva_vma_days_from_biopsy = "HVA/VMA days from biopsy"

In [23]:
clinical_data = clinical_data.fillna({hva_vma_days_from_biopsy: pd.NA})

In [24]:
for c in ["17q gain", "11q loss/LOH", "7q gain", "1p loss/LOH", "ALK"]:
    clinical_data[c] = clinical_data[c].str.rstrip().str.lstrip().str.capitalize()

In [25]:
clinical_data = clinical_data.replace(
    to_replace={
        "17q gain": {
            "Yes  (wc, relative, 4n)": "Yes|WC|relative|4N",
            "Yes  (relative, 5n)": "Yes|relative|5n",
            "Yes , (relative, (5n)": "Yes|relative|5N",
            "Yes, relative, 4n)": "Yes|relative|4N",
            "Yes (wc, relative 5n)": "Yes|WC|relative|5N",
            "Yes (wc, relative 4n)": "Yes|WC|relative|4N",
            "Yes (relative wc, 4-5n)": "Yes|WC|relative|4N|5N",
            "Yes (relative, wc, 6n)": "Yes|WC|relative|6N",
            "Yes, 4n (relative)": "Yes|relative|4N",
            "Yes, wc, relative, 4n)": "Yes|WC|relative|4N",
            "Yes (6n)": "Yes|6N",
            "Yes (wc, relative, 4n)": "Yes|WC|relative|4N",
            "Yes (wc, relatve, 4n)": "Yes|WC|relative|4N",
            "Yes (relative, 4n)": "Yes|relative|4N",
            "Yes (wc relative gain)": "Yes|WC|relative|gain",
            "Yes (wc, 4n)": "Yes|WC|4N",
            "Yes 9wc, relative, 4n)": "Yes|WC|relative|4N",
            "Yes (wc, releative, 4n)": "Yes|WC|relative|4N",
        },
        "7q gain": {
            "No ": "No",
            "Yes ": "Yes",
            "Yes (wc)": "Yes|WC",
            "yes (wc, relative, 4n)": "Yes|WC|relative|4N)",
            "Yes": "Yes",
            "Yes (relative, 6n)": "Yes|relative|6N",
            "No": "No",
            "Yes (wc, relagtive 5n)": "Yes|WC|relative|5N",
            "Yes (relative, 4-5n)": "Yes|relative|4N|5N|",
            "Yes (wc, relative, 4n)": "Yes|WC|relative|4N",
            "Yes, wc, relative, 4n)": "Yes|WC|relative|4N",
            "Yes (wc, 4n)": "Yes|WC|4N",
            "Yes (relative, 4n)": "Yes|relative|4N",
            "Yes 9wc, relative, 4Nn": "Yes|WC|relative|4N",
            "Yes (wc, releative, 4n)": "Yes|WC|relative|4N",
            "Yes (wc, relative, 4n) ": "Yes|WC|relative|4N",
            "Yes (wc relative gain)": "Yes|WC|relative|gain",
            "Yes 9wc, relative, 4n)": "Yes|WC|relative|4N",
        },
        "1p loss/LOH": {
            "No": "No",
            "Yes (relative, 2n)": "Yes|relative|2N",
            "Yes (relative, 2n, cnloh)": "Yes|relative|2N|cnLOH",
            "Yes": "Yes",
            "Yes (wc)": "Yes|WC",
            "No?": "No",
        },
        "11q loss/LOH": {
            "No": "No",
            "Yes": "Yes",
            "Yes, (cn neutral loh)": "Yes|neutral cnLOH",
            "Yes (relative, 2n, cnloh)": "Yes|relative|2N|cnLOH",
            "Yes (deletion)": "Yes|deletion",
            "Yes (relative, 2n, wc, cn loh)": "Yes|relative|WC|2N|cnLOH",
            "Yes (wc)": "Yes|WC",
            "Yes (relative, wc, 2n)": "Yes|relative|WC|2N",
            "Yes (wc, relative loss)": "Yes|relative|WC",
            "Yes (cn neutral loh)": "Yes|neutral|cnLOH",
        },
        "ALK": {
            "Wt": "WT",
            "F1245l (somatic)": "F1245L|somatic",
            "Wt (alk gain)": "WT|ALK gain",
            "Wt (phox2b wt)": "WT|Phox2B WT",
            "Wt/phox2b with a heterozygous polyalanine expansion (20/33).": "WT|Phox2B with a heterozygous polyalanine expansion (20/33)",
            "N/a": pd.NA,
            "Arg1275gln": "Arg1275Gln",
            "Wt (diagnosis and this specimen)": "WT",
            "wt": "WT",
            "WT (PHOX2b WT)": "WT|Phox2B WT",
            "Wt / phox2b wt": "WT|Phox2B WT",
            "F1174l": "F1174L",
        },
        "Other mutations (source)": {
            "none (FoundationOne)": pd.NA,
            "NUDT15 (NM_018283.2), c.415C>T (p.Arg139Cys)": "NUDT15 (NM_018283.2)|c.415C>T (p.Arg139Cys)",
            "BRAF Gly469Ala (CHOP NGS)": "BRAF Gly469Ala|CHOP NGS",
        },
        "Genomic studies done": {
            "SNP array, ALK seq, MYCN FISH": "SNP array|ALK seq|MYCN FISH",
            "SNP, ALK seq": "SNP array|ALK seq",
            "SNP array, ALK seq": "SNP array|ALK seq",
            "SNP, ALK/PHOX2B sequencing ": "SNP array|ALK/PHOX2B sequencing",
            "CHOP NGS, SNP array": "CHOP NGS|SNP array",
            "CHOP NGS": "CHOP NGS",
            "SNP array": "SNP array",
            "SNP array, ALK seq (tumor and germline)": "SNP array|ALK seq (tumor and germline)",
            "SNP array, ALK seq ": "SNP array|ALK seq",
            "SNP, ALK seq, Foundation one (no mutations) ": "SNP array|ALK seq|Foundation one (no mutations)",
            "SNP array (post chemo), ALK seq": "SNP array (post chemo)|ALK seq",
            "SNP array, CHOP NGS": "SNP array|CHOP NGS",
            "b/l SNP and CHOP NGS": "b/l SNP array|CHOP NGS",
        },
    },
)

In [26]:
clinical_data["17q gain"] = clinical_data["17q gain"].astype("category")
clinical_data["7q gain"] = clinical_data["7q gain"].astype("category")
clinical_data["1p loss/LOH"] = clinical_data["1p loss/LOH"].astype("category")
clinical_data["ALK"] = clinical_data["ALK"].astype("category")
clinical_data["Other mutations (source)"] = clinical_data["Other mutations (source)"].astype("category")

In [27]:
for c in [
    "INSS stage",
    "INRG stage",
    "Ploidy value",
    "MKI",
    "Degree of differentiation",
    "Histolgic classification - INPC",
]:
    clinical_data[c] = clinical_data[c].str.rstrip().str.lstrip().str.capitalize()

In [28]:
clinical_data["Degree of differentiation"] = clinical_data["Degree of differentiation"].replace(
    "\n|\xa0", "", regex=True
)

In [29]:
clinical_data = clinical_data.rename(mapper={"Histolgic classification - INPC": "Histologic classification - INPC"})

In [30]:
clinical_data = clinical_data.replace(
    to_replace={
        "INSS stage": {"4s": "4S", "2a": "2A", "2b": "2B", "2b??": "2B"},
        "INRG stage": {
            "M": "M",
            "L2": "L2",
            "Ms": "MS",
            "M (from diagnosis)": "M",
            "L1": "L1",
        },
        "Ploidy value": {
            "Hyperdiplod (3-4n)": "Hyperdiploid|3N|4N",
            "Diploid": "Diploid",
            "Diploid (diagnosis)": "Diploid",
            "Hyperdiploid (3-4n)": "Hyperdiploid",
            "Hyeperdiplod (3n)": "Hyperdiploid|3N",
            "Hyperdiploid (3n)": "Hyperdiploid|3N",
            "Hyperdiploid (3-4n) with scas": "Hyperdiploid|3N|4N",
            "Hyperdiploid (3n of 10 chromosomes)": "Hyperdiploid|3N",
            "Hyperdipoid (3n)": "Hyperdiploid|3N",
            "Hyperdiploid (3n) w/ scas": "Hyperdiploid|3N",
            "Diploid (hyperdiploid at diagnosis?)": "Diploid",
            "Hyperdip)loid (3n)": "Hyperdiploid|3N",
            "Hyperidiploid/with scas": "Hyperdiploid",
            "Hyperploid (3n)": "Hyperdiploid|3N",
            "Hyperdiploid": "Hyperdiploid",
            "Hyperdiploid (near 4n)": "Hyperdiploid|4N",
            "Hyeperdiplid (3n)": "Hyperdiploid|3N",
        },
        "MKI": {
            "Intermediate": "Intermediate",
            "Low": "Low",
            "High": "High",
            "High (diagnostic)": "High",
            r"Low (<1\%, diagnostic)": "Low",
            "Intermediate (diagnosis)": "Intermediate",
            "Low/intermediate": "Intermediate",
            "Diagnosis = low": "Low",
            "High (from diagnosis)": "High",
            "Low (diagnosis)": "Low",
            "Low (diagnostic)": "Low",
            "High (and one clone with low)": "High",
        },
        "Degree of differentiation": {
            "Poorly differentiated": "Poorly differentiated",
            "Poorly differentiated/status-post treatment, diagnosis = poorly diffeentiated": "Poorly differentiated",
            "Residual neuroblastoma with effects from therapy including variable differentiation of neuroblastic cells with increased nodularity secondary to fibrosis and decreased cellularity overall. dystrophic calcification, foci of hemosiderin pigment and lymphoid aggregates are present. the neuroblastic cells are notable for nucleoli (poorly differentiated = diagnosis)": "Poorly differentiated",
            "Poorly differentiated (many large anaplastic tumor cells)": "Poorly differentiated",
            r"A peripheral neuroblastic tumor, post treatment. the majority of this tumor (>80\%) consists of viable neuroblastoma, the predominant pattern of which appears consistent with a differentiating neuroblastoma at this time. the remainder of the tumor mass consists of mostly fibrosis with foci of hemosiderin (diagnosis = poorly differentiated)": "Differentiating",
            "Poorly differentiated, chemotherapeutic effect, no tumor necrosis": "Poorly differentiated",
            "Posty-treatment, multiple lobules of viable neuroblastic tumor. there is focal gangliocytic differentiation with no schwannian stroma. extensive necrosis with hemorrhage, hemosiderin and calcification accounts for approximately 30 percent of the lesional tissue (diagnosis = poorly differentiated)": "Poorly differentiated",
            "Treatment effect including widespread calcifications, hemorrhage and hemosiderin deposition with up to 40% tumor necrosis. remaining viable tumor has the appearance of poorly differentiated neuroblastoma similar to that seen in the patient's prior biopsy (poorly differentiated = diagnosis)": "Poorly differentiated",
            "(post-treatment, limited necrosis, diagnosis = poorly differentiated)": "Poorly differentiated",
            "Poorly diffeentiated": "Poorly differentiated",
            "Differentiating (diagnosis = poorly differentiated)": "Poorly differentiated",
            "Maturational changes typical of treated neuroblastoma, with fields of ganglioneuroma, ganglioneuroblastoma, and foci of differentiating neuroblastoma (diagnostic = differentiating)": "Differentiating",
            "Undifferentiated (with extensive necrosis)": "Undifferentiated",
            "Extensive differentiation, with focal calcification and no significant necrosis. the majority is composed of differentiating neuroblastoma with areas of intermixed ganglioneuroblastoma and poorly differentiated neuroblastoma": "Differentiating",
            "The tumor resembles a stroma poor, poorly differentiated neuroblastoma with areas of necrosis, calcification and lymphocyte infiltration. there is focal fibroplasia with abundant pigment interpreted as treatment effect.": "Poorly differentiated",
            "Differentiating neuroblastoma, wth focus of hemmorhage maf calcification, with nodule of poorly differentiated nb, areas of ganglioneuroblastomatpus nad ganglioneuroamatous hisdtology, almost all viable tumor with little necrosis (diagnosis = poorly differentiated)": "Poorly differentiated",
            r"Residual nb, consistent with treatment effect, approx 10\% viable tumor, (diagnosis = poorly differentiated)": "Poorly differentiated",
            'Post chemotherapy, with minimal      treatment effect, the current specimen, post-chemotherapy, is very similar in appearance: poorly differentiated neuroblastoma with less than 30% tumor necrosis. the largest tumor dimension is 5 cm, and there is a "shell" of ganglioneuromatous tissue at the periphery of the malignant tumor, which highly suggests that this tumor originated as a nodular ganglioneuroblastoma (diagnosis = poorly differentated)': "Poorly differentiated",
            "Post-treatent nb, dystrophic calcification, neuropil wth neruoblasdts and maturing ganglion cells, (diagnosis = poorly differentiated)": "Poorly differentiated",
            r"Post-treatment nb with minimal necrosis (<5\%), nodules of neoplastic cells with varying ganglionic differentation, (diagnosis =poorly differentiated)": "Poorly differentiated",
            r"Poorly differentiated (biphasic appearance: outer differentiating w/ <50\% schwannian stroma, inner hemorrhagic  and poorly differentiated ? nodular ganglioneuroblastoma)": "Poorly differentiated",
            "Post-treatment - mass predominantly composed of neuropil and mature/maturing ganglion cells. schwannian-like stroma is present, mostly at the periphery. dystrophic calcifications are present (diagnosis = differentiating)": "Differentiating",
            "Poorly differentiated (nodular ganglioneuroblastoma)": "Poorly differentiated",
            "Differentiating": "Differentiating",
            "Poorly dfferentiated/differentiating , complex": "Poorly differentiated",
            "Peripheral neuroblastic tumor, status post-chemotherapy. note: sections show largely viable tumor with the appearance of maturing neuroblastoma with areas of fibrosis, hemorrhage and scattered foci of dystrophic calcifications. no significant tumor necrosis is identified.poorly (diagnosis=differentated)": "Differentiating",
            "Pooorly differentiated": "Poorly differentiated",
            r"Nb with chemo changes- variety of differentation - 15\% of tumor (diagnostic = poorly differentiated)": "Poorly differentiated",
            "Differentiating, status post treatment, now with features of intermixed (schwannian stroma-rich) ganglioneuroblastoma. the tumor is intermixed with and surrounded by a dense collagen, adipose tissue and skeletal muscle (diagnostic = differentiating)": "Differentiating",
            "Poorly differentiated (diagnosis)": "Poorly differentiated",
            r"Well-differentiated, s/p chemo, nests of neuroblasts showing extensive differentiation towards ganglion cells and neuropil (50\% of tumor cells), tumor necrosis (15\%),  (diagnostic = differentiating)": "Differentiating",
        },
        "Histolgic classification - INPC": {
            "Favorable histology": "Favorable",
            "Favorbale histology, diagnosis = favorable histology": "Favorable",
            "Unfavorable histology": "Unfavorable",
            "Unfavorable histology (diagnosis)": "Unfavorable",
            "Favorable (diagnosis)": "Favorable",
            "Diagnosis = unfavorable histology": "Unfavorable",
            "Unfavorable histology (from diagnosis)": "Unfavorable",
            "Favorable histology (diagnosis)": "Favorable",
            "Favorable histology (diagnostic)": "Favorable",
            "N/a": pd.NA,
            "Unfavorable histology (diagnostic tumor)": "Unfavorable",
            "Unfavorable histology = diagnosis": "Unfavorable",
            "Unfavorable histology (nodular gangloneuroblastoma with a poorly differentiated neuroblastic component)": "Unfavorable",
            "Diagnosis = favorable histology": "Favorable",
            "Favorable (diagnostic)": "Favorable",
            "Unfavorable": "Unfavorable",
            "Unfavorable hiostology (diagnostic)": "Unfavorable",
            "Favorable  (diagnosis)": "Favorable",
            "Unfavorbale histology": "Unfavorable",
        },
        "Genomics source": {
            "This specimen ": "This specimen",
            "Diagnostic specimen": "Diagnostic specimen",
            "none": pd.NA,
            "this specimen": "This specimen",
            "Diagnostic tumor ": "Diagnostic specimen",
            "This specimen": "This specimen",
            "This specimen?": "This specimen",
        },
        # "MYCN amplification": {
        #     "No ": "False",
        #     "No": "False",
        #     "Yes": "True",
        # },
    },
)
clinical_data["MYCN amplification"] = clinical_data["MYCN amplification"].map(lambda x: False if "No" in x else True)

### Final Column Renaming

In [31]:
clinical_data

BuckarooWidget(buckaroo_options={'sampled': ['random'], 'auto_clean': ['aggressive', 'conservative'], 'post_pr…

In [32]:
# Rename Columns
clinical_data = clinical_data.rename(
    columns={
        "Race": "Ethnicity",
        "Biopsy/surgery location": "Tissue",
        "11q loss/LOH": "11q LOH",
        "1p loss/LOH": "1p LOH",
        "Classification of specimen": "Classification",
    },
)

In [33]:
# Convert some columns to categoricals
for c in [
    "Paired sequence",
    "Sex",
    "Classification",
    "Ethnicity",
    "Tissue",
    "Risk",
    "INSS stage",
    "INRG stage",
    "Ploidy value",
    "MKI",
    "Histolgic classification - INPC",
    "Genomics source",
    "MYCN amplification",
    "17q gain",
    "7q gain",
    "1p LOH",
    "11q LOH",
    "ALK",
    "Other mutations (source)",
    "Genomic studies done",
    "treatment btw biopsies",
]:
    clinical_data[c] = clinical_data[c].astype("category")

In [34]:
clinical_data = clinical_data.fillna(value=pd.NA)

### Add Clinical Data to LaminDB

In [35]:
clinical_artifact = ln.Artifact.from_df(
    df=clinical_data,
    key="clinical_data.parquet",
    run=run,
    description="Contains sample level clinical data",
    version="1",
)
clinical_artifact.save(upload=True)

Artifact(uid='itukWOpYdOI7thfDJA36', version='1', description='Contains sample level clinical data', key='clinical_data.parquet', suffix='.parquet', type='dataset', _accessor='DataFrame', size=38440, hash='ZIDdlIAE0Ikxj03Z9RmMZg', _hash_type='md5', visibility=1, _key_is_virtual=True, created_by_id=1, storage_id=1, transform_id=1, run_id=1, updated_at='2024-08-12 17:02:03 UTC')

In [36]:
tissues: list[Registry] = bt.Tissue.from_values(clinical_data["Tissue"])
ln.save(tissues)

In [37]:
ethnicities: list[Registry] = bt.Ethnicity.from_values(clinical_data["Ethnicity"])
ln.save(ethnicities)

## Channel Validation

In [38]:
sample_fov_markers = set(ns.natsorted(x.stem for x in fov_dir.glob("*/*.tiff")))

In [39]:
public_cm = bt.CellMarker.public()

inspected_markers = public_cm.inspect(values=sample_fov_markers, field=public_cm.name)

❗ 14 terms (29.80%) are not validated for name: chan_48, Noodle, Fe, NCAML1, chan_71, chan_45, TRKA, Au, GPC2, chan_145, pS6, chan_39, TRKB, HH3dsDNA


In [40]:
standardized_markers_mapper = public_cm.standardize(values=sample_fov_markers, field=public_cm.name, return_mapper=True)

In [41]:
copied_markers = [
    standardized_markers_mapper[m] if m in standardized_markers_mapper.keys() else m for m in sample_fov_markers.copy()
]

In [44]:
inspected_markers2 = public_cm.inspect(values=copied_markers, field=public_cm.name)
manually_added_markers = [bt.CellMarker(name=n) for n in inspected_markers2.non_validated]

❗ 14 terms (29.80%) are not validated for name: chan_48, Noodle, Fe, NCAML1, chan_71, chan_45, TRKA, Au, GPC2, chan_145, pS6, chan_39, TRKB, HH3dsDNA


In [45]:
valdiated_markers = bt.CellMarker.from_values(values=inspected_markers2.validated, field="name")

In [46]:
ln.save(valdiated_markers)
ln.save(manually_added_markers)

In [47]:
clinical_data = ln.Artifact.filter(key__icontains="clinical_data").one().load()

In [48]:
clinical_artifact.describe()

Artifact(uid='itukWOpYdOI7thfDJA36', version='1', description='Contains sample level clinical data', key='clinical_data.parquet', suffix='.parquet', type='dataset', _accessor='DataFrame', size=38440, hash='ZIDdlIAE0Ikxj03Z9RmMZg', _hash_type='md5', visibility=1, _key_is_virtual=True, updated_at='2024-08-12 17:02:03 UTC')
  Provenance
    .created_by = 'srivarra'
    .storage = '/Users/srivarra/Davis Lab/neuroblastoma/nbl/data/db'
    .transform = 'Clean Clinical Data'
    .run = '2024-08-12 17:01:42 UTC'



### Finishing up

In [50]:
ln.finish()

❗ cells [(1, 1), (2, 4), (41, 44)] were not run consecutively
